In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================
# All imports consolidated here for the block synchronization pipeline.
# Helper functions previously defined inline in Cell 1 now live in
# `eye_tracking_system_tools.preprocessing.notebook_helpers` so the
# Preprocessing GUI can import them too.

from __future__ import annotations
from pathlib import Path

import numpy as np
import pandas as pd

# Project utilities
from eye_tracking_system_tools.preprocessing import utility_functions as uf
from eye_tracking_system_tools.preprocessing.notebook_helpers import (
    simple_sync_build,
    describe_eye_tick,
    shift_eye_df_by_index,
    build_arena_grid_df,
    build_final_sync_df_merge_nearest,
    verify_final_df_against_sources,
    export_final_sync_df,
    load_final_sync_df,
    plot_simple_sync_bokeh,
    hover_inspect_eyes_bokeh,
    sanity_plot_final_df,
    insert_dup_by_pos,
    insert_dup_by_oe_sample,
    find_jittery_frames,
    add_intermediate_elements,
    export_eye_data_2d,
    create_distance_plot,
    bokeh_plotter,
)

# Optional visualization (for jitter analysis plots)
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# All synchronization helper functions are now defined in
# eye_tracking_system_tools.preprocessing.notebook_helpers
# and imported in Cell 0. This cell is kept empty to preserve the
# numbering used in the original notebook (saved outputs, anchor
# links, etc.). See notebook_helpers.py for the verbatim source.


# Block Synchronization Pipeline

This notebook implements the complete block synchronization workflow, bringing eye tracking data from raw videos to synchronized `left_eye_data` and `right_eye_data` DataFrames ready for downstream analysis.

## Workflow Overview

1. **Setup**: Initialize BlockSync object
2. **Data Preparation**: Handle videos, parse Open Ephys events, extract brightness
3. **Arena Grid**: Build 60Hz master grid from arena TTL events
4. **Simple Synchronization**: Initial eye sync using first TTL anchor
5. **Manual Correction**: Interactive alignment with LED events
6. **Final Merge**: Merge corrected eyes onto arena grid
7. **Verification**: Check alignment quality
8. **Jitter Correction**: Remove camera jitter artifacts
9. **LED Blink Removal**: Remove LED blink artifacts
10. **Final Export**: Create left/right_eye_data for downstream use

---

## Step 1: Block Setup and Initialization

In [ ]:
# block instantiation:
bad_blocks = [] #
experiment_path = Path(r"D:\sample_data_for_eye_repo")

block_numbers = [15]
animal = 'PV_106'
block_collection = uf.block_generator(block_numbers=block_numbers,
                                      experiment_path=experiment_path,
                                      animal=animal,
                                      bad_blocks=bad_blocks)
for block in block_collection:
    block.channeldict = None
    if block.animal_call == 'PV_208':
        block.channeldict={1: 'LED_driver',
                           7: 'L_eye_TTL',
                           2: 'Arena_TTL',
                           8: 'R_eye_TTL'}
    elif block.animal_call == "TE_21":
        block.channeldict={1:'Arena_TTL',
                           4:'LED_driver',
                           5:'R_eye_TTL',
                           8:'L_eye_TTL'}
# create a block_dict object for ease of access:
block_dict = {}
for b in block_collection:
    block_dict[str(b.block_num)] = b

## Step 2: Data Preparation

Run the following methods to prepare the block data:
- `handle_eye_videos()`: Convert and validate eye video files
- `parse_open_ephys_events()`: Parse Open Ephys events (includes manual TTL selector for non-standard paradigms)
- `handle_arena_files()`: Process arena video files
- `get_eye_brightness_vectors()`: Extract brightness values from eye videos

In [ ]:
block = block_collection[0]
block.handle_eye_videos()
block.parse_open_ephys_events()
block.handle_arena_files()
block.get_eye_brightness_vectors()


## Step 3: Build Arena Grid

Create the master 60Hz grid from arena TTL events. This grid will be used to align all streams. The function automatically handles cases where arena fps differs from 60Hz by creating a pseudo-60Hz grid.

In [ ]:
# after block.parse_open_ephys_events() succeeded (manual or auto)
arena_grid_df, info = build_arena_grid_df(block, target_fps=60.0, arena_fps_tol_hz=5.0)

# info.used_pseudo_60hz tells you whether it applied the correction
print(info)

## Step 4: Simple Eye Synchronization

Build per-eye DataFrames using the simple anchor-at-first-TTL approach. This creates initial synchronization that can be manually corrected in the next step.

The algorithm:
1. Read internal timestamps for each eye
2. Get the FIRST TTL sample for that eye
3. Place frame 0 at that TTL, others by internal timing deltas
4. Attach brightness values

In [ ]:
# Build and save the per-eye DataFrames
dfL, dfR = simple_sync_build(block, export=True)

# Quick look and manual correction to LED grid
plot_simple_sync_bokeh(block, dfL, dfR, show_led=True)


In [ ]:
# Inspect ms per tick (helps translate slider “ticks” to ms)
print("Left tick ≈ %.3f ms"  % describe_eye_tick(dfL))
print("Right tick ≈ %.3f ms" % describe_eye_tick(dfR))

## Step 5: Manual Synchronization Correction

Use the interactive Bokeh plot to visually align the eye traces with LED events. Adjust sliders to shift traces, then apply corrections using `shift_eye_df_by_index()`.

**Note**: The plot opens in your default browser. Use the sliders to find the correct shift values, then apply them in the next cell.

In [ ]:
# Use the function to apply shift, 
# Notice: the values here should be inverse to the shift which corrects the interactive plot
dfL_shifted = shift_eye_df_by_index(dfL,-3)
dfR_shifted = shift_eye_df_by_index(dfR,9)

## Step 6: Verification & optional drift-shift

Verify the final synchronization by plotting.
If required, optional dropped frame correction approach is available via frame duplication and dataframe shifts

In [ ]:
# use this function to verify shift implemented
plot_simple_sync_bokeh(block, dfL_shifted, dfR_shifted, show_led=True)

In [ ]:
# Optional: Hover inspection plot for detailed frame-by-frame inspection
# hover_inspect_eyes_bokeh(dfL_shifted, dfR_shifted)


In [ ]:
# Optional: Insert duplicate frames to correct for dropped frames
# Example usage (uncomment if needed):
# dfL_fix = insert_dup_by_pos(dfL_shifted, [3], duplicate="prev", leave_trailing_nan=True)
# dfR_fix = insert_dup_by_pos(dfR_shifted, [3], duplicate="prev", leave_trailing_nan=True)



## Step 7: Merge onto Arena Grid

Merge the corrected eye DataFrames onto the 60Hz arena grid using nearest-neighbor matching with tolerance. This creates the final synchronized dataframe (`final_sync_df`) used downstream.

In [ ]:
# Merge onto the 60 Hz arena grid with nearest-with-tolerance (no resort of eye dfs)
final_df = build_final_sync_df_merge_nearest(
    block, dfL_shifted, dfR_shifted,
    target_fps=60.0,
    tol_frac=0.90,      # accept nearest within 90% of a 60 Hz tick; tune 0.7–1.2 if needed
    pre_shift_left=0,   # IMPORTANT: already pre-shifted
    pre_shift_right=0,
    export_csv=True
)

## Step 8: Final Sync Verification
builds the brigtness grid but from final_df, if everything went according to plan you should get perfect alignment between eyes and with the LED driver off rising edges

In [ ]:

fs = float(block.sample_rate)
sanity_plot_final_df(final_df, fs, show_led_off=True, led_off_samples=block.oe_events['LED_driver'].dropna().astype(int).to_numpy())


## Step 9: Statistics-based verification and export


In [ ]:
# dfL_s/dfR_s are the eye DFs you actually shifted (exact slider semantics)
stats = verify_final_df_against_sources(block, final_df, dfL_shifted, dfR_shifted, target_fps=60.0, tol_frac=0.9)


In [ ]:
# Export the final synchronized dataframe
export_final_sync_df(block, final_df=final_df, overwrite=True)

### Preprocessing of synchronized data from here (loads final_sync from folder if available)

In [ ]:

for block in block_collection:
    load_final_sync_df(block)


In [ ]:
for block in block_collection:
    # load relevant data
    block.handle_eye_videos()
    block.parse_open_ephys_events()
    block.handle_arena_files()
    block.get_eye_brightness_vectors()
    load_final_sync_df(block)
    # read deeplabcut annotations and construct ellipse parameters per video frame
    block.read_dlc_data(overwrite=False, export=True)

In [ ]:
for block in block_collection:
    # cross-corr distance estimation between eye video frames to correct for jitter
    # (relatively slow, but only needs to be run once per block, will load from disk if available)
    block.get_jitter_reports(export=True, overwrite=False, remove_led_blinks=False, sort_on_loading=True)

In [ ]:
# perform jitter correction and remove led blinks
for block in block_collection:
    block.correct_jitter()
    block.find_led_blink_frames(plot=True)
    block.remove_led_blinks_from_eye_df(export=True)

In [ ]:
df_inds_to_remove_l, vid_inds_l = find_jittery_frames(block, 'left', max_distance=60, diff_threshold=5,
                                                      gap_to_bridge=24)
df_inds_to_remove_r, vid_inds_r = find_jittery_frames(block, 'right', max_distance=60, diff_threshold=5,
                                                      gap_to_bridge=24)

# These are verification plots for the jitter outlier removal functions:
# to verify, I want a bokeh explorable:
rdf = pd.DataFrame.from_dict(block.re_jitter_dict)
ldf = pd.DataFrame.from_dict(block.le_jitter_dict)

In [ ]:
# visualize right eye
uf.bokeh_plotter([rdf.top_correlation_dist], ['drift_distance'], peaks=vid_inds_r)

In [ ]:
# visualize left eye
uf.bokeh_plotter([ldf.top_correlation_dist], ['drift_distance'], peaks=vid_inds_l)

In [ ]:
# if you are happy with the results, remove the outliers
block.remove_eye_datapoints_based_on_video_frames('right', indices_to_nan=vid_inds_r)
block.remove_eye_datapoints_based_on_video_frames('left', indices_to_nan=vid_inds_l)

In [ ]:
# create the eye dataframes, integrating the cleaned le/re dataframes with the ellipse parameters
# this will create block.left_eye_data and block.right_eye_data
for block in block_collection:
    block.create_eye_data()

In [ ]:

# Export the final eye dataframes
for block in block_collection:
    export_eye_data_2d(block)